# Delivery360 governance demo

Run `05_governance.py` first. This notebook is optional: it changes only the current user's mapping, demonstrates full versus restricted access, and restores the administrator baseline at the end.

Removing SELECT is shown only as SQL text because revoking access from the active user can interrupt this notebook.

In [ ]:
dbutils.widgets.text('catalog', 'delivery360')
catalog = dbutils.widgets.get('catalog')
governance = f'`{catalog}`.`governance`'
gold = f'`{catalog}`.`gold`'
current_user = spark.sql('SELECT current_user() AS user').first()['user']
safe_user = current_user.replace("'", "''")
print(f'Demo identity: {current_user}')

## 1. Inspect the baseline

In [ ]:
display(spark.table(f'{governance}.profile_access_matrix'))
display(spark.table(f'{governance}.user_region_access').where(f"user_email = '{current_user}'"))
display(spark.table(f'{governance}.column_access').where(f"user_email = '{current_user}'"))
display(spark.sql(f'SHOW GRANTS ON TABLE {gold}.fact_orders'))

## 2. Full-access state

The current user sees all regions and unmasked PII.

In [ ]:
spark.sql(f"DELETE FROM {governance}.user_region_access WHERE user_email = '{safe_user}'")
spark.sql(f"DELETE FROM {governance}.column_access WHERE user_email = '{safe_user}'")
spark.sql(f"INSERT INTO {governance}.user_region_access SELECT '{safe_user}', '*', 'admin'")
spark.sql(f"INSERT INTO {governance}.column_access SELECT '{safe_user}', 'admin', true")
display(spark.sql(f'SELECT order_id, region, email, phone, order_total FROM {gold}.fact_orders LIMIT 10'))

## 3. Restricted-access state

The user keeps SELECT, but sees only San Jose and PII is masked.

In [ ]:
spark.sql(f"DELETE FROM {governance}.user_region_access WHERE user_email = '{safe_user}'")
spark.sql(f"DELETE FROM {governance}.column_access WHERE user_email = '{safe_user}'")
spark.sql(f"INSERT INTO {governance}.user_region_access SELECT '{safe_user}', 'San Jose', 'operations'")
spark.sql(f"INSERT INTO {governance}.column_access SELECT '{safe_user}', 'operations', false")
display(spark.sql(f'SELECT order_id, region, email, phone, order_total FROM {gold}.fact_orders ORDER BY order_id LIMIT 20'))

## 4. Optional permission test

Do not revoke SELECT from your own user. Use a separate demo group or second identity. Revoking SELECT produces permission denied; it does not produce masked data.

In [ ]:
demo_principal = 'delivery360_demo_readers'
print(f'GRANT USE CATALOG ON CATALOG {catalog} TO `{demo_principal}`')
print(f'GRANT USE SCHEMA ON SCHEMA {catalog}.gold TO `{demo_principal}`')
print(f'GRANT SELECT ON TABLE {catalog}.gold.fact_orders TO `{demo_principal}`')
print(f'REVOKE SELECT ON TABLE {catalog}.gold.fact_orders FROM `{demo_principal}`')

## 5. Roll back the demo

In [ ]:
spark.sql(f"DELETE FROM {governance}.user_region_access WHERE user_email = '{safe_user}'")
spark.sql(f"DELETE FROM {governance}.column_access WHERE user_email = '{safe_user}'")
spark.sql(f"INSERT INTO {governance}.user_region_access SELECT '{safe_user}', '*', 'admin'")
spark.sql(f"INSERT INTO {governance}.column_access SELECT '{safe_user}', 'admin', true")
print('Rollback complete: current user restored to admin / all regions / PII visible')
display(spark.sql(f'SELECT order_id, region, email, phone FROM {gold}.fact_orders LIMIT 10'))